In [5]:
"""
This database is a representation of a UK hotel chain, Grand Stay Hotels, comprised of five tables related to it.
 The table Hotels contains 15 properties in UK cities with the star rating and the price range. There are 195 room records in the Rooms
  table which include room type, floor, nightly rate and availability of the sea view. The personal guest information in the form of 1,000 rows of names,
  gender, nationality, loyalty level, and satisfaction level is stored in the table named as the Guests. The table of bookings is used to store
  1,500 stays between guests and rooms that have a check-in and check-out date, mode of payment, channel of booking and bill amount. Stay-Services table
  records other services like dining, spa and laundry and has a composite primary key of booking and service id as they cannot have the same service per booking.

Three traces of missing data were also induced intentionally to represent real data quality problems. Approximately 8 percent of the guests do not
have an email address, 15 percent do not have a passport number and 10 percent do not have a satisfaction score. Three more duplicate guest profiles were
 also deliberately added to represent the case of guests reserving using two distinct channels and receiving two distinct records.
 """
# At the beginning we are importing the libraries which are compulsory for this task.

import sqlite3
import random
import os
from datetime import date, timedelta

random.seed(53)   # picked 53 so results differ from anyone using 42 or 19



# First names split by gender so generated guests feel like real people by queries and python script

MALE_NAMES   = ["James","Oliver","Harry","George","Noah","Ethan","Finn",
                "Rory","Nathan","Dylan","Hugo","Felix","Marcus","Leo",
                "Eli","Connor","Hamza","Arjun","Omar","Tariq","Luca","Theo"]

FEMALE_NAMES = ["Olivia","Emma","Amelia","Grace","Lily","Ella","Evie",
                "Isla","Chloe","Sophie","Freya","Zoe","Nora","Clara",
                "Iris","Ada","Luna","Priya","Fatima","Layla","Mia","Aria"]

# Gender neutral names used for Non-binary and Prefer not to say guests
OTHER_NAMES  = ["Alex","Jordan","Taylor","Morgan","River","Casey","Sam","Quinn"]

# Surnames reflect the actual demographic mix of UK residents
# Includes common British, South Asian, Arabic, and Irish surnames
SURNAMES     = ["Smith","Jones","Williams","Brown","Taylor","Davies","Evans",
                "Wilson","Roberts","Johnson","Walker","Wright","Robinson",
                "Thompson","Green","Hall","Lewis","Harris","Patel","Khan",
                "Singh","Ahmed","Ali","Murphy","Scott","Mitchell","Carter",
                "Reed","Cole","Ward","Barnes","Fisher","Dixon","Malik","Gupta"]

# Email providers that real UK guests would actually use
# Kept to well known services so the addresses look genuine
EMAIL_HOSTS  = ["gmail.com","yahoo.co.uk","hotmail.com","outlook.com",
                "icloud.com","protonmail.com","fastmail.com","me.com"]

# A sample of real UK postcodes covering different cities and regions
# London, Manchester, Edinburgh, Bristol, Cardiff, Leeds etc.
# Used only for UK  post codes
UK_POSTCODES = ["SW1A 1AA","EC1A 1BB","W1A 0AX","M1 1AE","B1 1BB","LS1 1BA",
                "E1 6AN","N1 9GU","SE1 7PB","WC2N 5DU","BS1 5TR","CB2 1TN",
                "OX1 1DP","EH1 1YZ","G1 1DX","L1 1JH","NE1 4ST","S1 2PP",
                "CF10 1EP","GL50 1BN","PE1 1HF","HG1 1AA","BN1 1AA","TR1 1AA"]

# (hotel_name, city, star_rating, base_nightly_rate_min, base_nightly_rate_max)
HOTEL_DATA = [
    ("The Grand Kensington",  "London",       5, 280, 550),
    ("Harbour View Brighton", "Brighton",     4, 120, 240),
    ("Royal Mile Suites",     "Edinburgh",    4, 130, 260),
    ("The Cotswold Manor",    "Cheltenham",   5, 220, 420),
    ("Lakeside Retreat",      "Windermere",   3, 80,  160),
    ("The Northern Quarter",  "Manchester",   4, 110, 220),
    ("Clifton Heights",       "Bristol",      3, 75,  150),
    ("Cathedral Close Hotel", "Canterbury",   4, 100, 200),
    ("Balmoral Gardens",      "Glasgow",      3, 70,  140),
    ("The Oxford Spires",     "Oxford",       5, 200, 380),
    ("Castle View Inn",       "York",         3, 65,  130),
    ("Seaside Pavilion",      "Bournemouth",  3, 70,  140),
    ("The Lanes Boutique",    "Bath",         4, 140, 280),
    ("Docklands Prestige",    "Liverpool",    4, 100, 210),
    ("Regency Park Hotel",    "Harrogate",    4, 120, 230),
]

ROOM_TYPES       = ["Single","Double","Twin","Deluxe Double","Suite","Penthouse"]
ROOM_TYPE_WGTS   = [20, 35, 20, 15, 8, 2]

ROOM_FLOORS      = [1, 2, 3, 4, 5, 6, 7, 8]

GENDER_OPTIONS   = ["Male","Female","Non-binary","Prefer not to say"]
GENDER_WEIGHTS   = [48, 48, 2, 2]

# Weighted so most guests are UK-based, some international
NATIONALITY_POOL = (["United Kingdom"] * 5
                    + ["United States","Germany","France","Australia",
                       "UAE","India","Canada","Japan","Italy"])

LOYALTY_TIERS    = ["Bronze","Silver","Gold","Platinum"]
LOYALTY_WEIGHTS  = [50, 28, 16, 6]    # most guests are Bronze, Platinum is rare

BOOKING_STATUSES   = ["Confirmed","Checked-in","Checked-out","Cancelled","No-show"]
BOOKING_STATUS_WGT = [15, 10, 60, 10, 5]

PAYMENT_OPTIONS  = ["Credit Card","Debit Card","PayPal","Bank Transfer","Voucher","Crypto"]
PAYMENT_WEIGHTS  = [42, 24, 18, 8, 5, 3]

BOOKING_CHANNELS = ["Website","Phone","Walk-in","OTA","Travel Agent"]
CHANNEL_WEIGHTS  = [45, 20, 10, 18, 7]    # OTA = Online Travel Agency like Booking.com

# In-stay services every hotel offers
SERVICE_CATALOGUE = [
    (1,  "Restaurant Dining",   15,  120),
    (2,  "Room Service",        10,  80),
    (3,  "Spa Treatment",       50,  180),
    (4,  "Minibar",             5,   45),
    (5,  "Laundry",             8,   40),
    (6,  "Airport Transfer",    25,  75),
    (7,  "Parking",             10,  30),
    (8,  "Business Centre",     15,  50),
    (9,  "Gym Access",          8,   20),
    (10, "Late Checkout",       20,  60),
    (11, "Early Check-in",      20,  60),
    (12, "Bicycle Rental",      12,  35),
]


# ============================================================
# The utility functions are small helper functions which help us  in the below ways.

# spin() selects an option randomly based on given weights. It is used when some outcomes should appear more often than others.
#  days_ago() creates a random past date within a given number of days and returns it in ISO format.
#  pick_forename() chooses a first name from a list based on the specified gender.
#  make_email() generates a realistic email using the guest’s name and a random number to avoid duplicates.
#  fake_passport() creates a UK-style passport number made of two letters followed by seven digits.

# ============================================================

def spin(options, weights):
    total   = sum(weights)
    landing = random.uniform(0, total)
    so_far  = 0
    for item, w in zip(options, weights):
        so_far += w
        if landing <= so_far:
            return item
    return options[-1]


def days_ago(furthest, closest=0):
        offset = random.randint(closest, furthest)
        return (date.today() - timedelta(days=offset)).isoformat()


def pick_forename(gender_str):
    """Returns a first name from the appropriate gender pool."""
    if gender_str == "Male":
        return random.choice(MALE_NAMES)
    elif gender_str == "Female":
        return random.choice(FEMALE_NAMES)
    else:
        return random.choice(OTHER_NAMES)


def make_email(first, last):
    tag  = random.randint(10, 9999)
    host = random.choice(EMAIL_HOSTS)
    return f"{first.lower()}.{last.lower()}{tag}@{host}"


def fake_passport():
    """
    Here this function will generate the passport number
    """
    letters = "ABCDEFGHJKLMNPRSTUVWXYZ"
    prefix  = random.choice(letters) + random.choice(letters)
    digits  = str(random.randint(1000000, 9999999))
    return prefix + digits


# function to create the tables for db schema creation and will try to remove  the extra 4 spaces from the def line only
#Also the below function will create all the tables in the selected db.
def create_all_tables(db_conn):
    db_conn.executescript("""

        CREATE TABLE IF NOT EXISTS Hotels (
            hotel_id        INTEGER  PRIMARY KEY AUTOINCREMENT,
            hotel_name      TEXT     NOT NULL UNIQUE,
            city            TEXT     NOT NULL,
            star_rating     INTEGER  NOT NULL CHECK(star_rating BETWEEN 1 AND 5),
            rate_min        REAL     NOT NULL CHECK(rate_min > 0),
            rate_max        REAL     NOT NULL CHECK(rate_max > rate_min)
        );

        CREATE TABLE IF NOT EXISTS Rooms (
            room_id         INTEGER  PRIMARY KEY AUTOINCREMENT,
            hotel_id        INTEGER  NOT NULL,
            room_number     TEXT     NOT NULL,
            floor_number    INTEGER  NOT NULL CHECK(floor_number >= 1),
            room_type       TEXT     NOT NULL
                            CHECK(room_type IN
                            ('Single','Double','Twin','Deluxe Double','Suite','Penthouse')),
            nightly_rate    REAL     NOT NULL CHECK(nightly_rate > 0),
            max_occupancy   INTEGER  NOT NULL CHECK(max_occupancy BETWEEN 1 AND 4),
            has_sea_view    INTEGER  NOT NULL DEFAULT 0 CHECK(has_sea_view IN (0,1)),
            FOREIGN KEY (hotel_id) REFERENCES Hotels(hotel_id)
        );

        CREATE TABLE IF NOT EXISTS Guests (
            guest_id          INTEGER  PRIMARY KEY AUTOINCREMENT,
            first_name        TEXT     NOT NULL,
            last_name         TEXT     NOT NULL,
            gender            TEXT     CHECK(gender IN
                              ('Male','Female','Non-binary','Prefer not to say')),
            birth_year        INTEGER  CHECK(birth_year BETWEEN 1930 AND 2005),
            age               INTEGER  CHECK(age BETWEEN 18 AND 95),
            nationality       TEXT     NOT NULL,
            email_address     TEXT     UNIQUE,
            phone_number      TEXT,
            postcode          TEXT,
            passport_number   TEXT     UNIQUE,
            loyalty_tier      TEXT     NOT NULL
                              CHECK(loyalty_tier IN ('Bronze','Silver','Gold','Platinum')),
            satisfaction_score INTEGER CHECK(satisfaction_score BETWEEN 1 AND 10),
            joined_programme  TEXT     NOT NULL
        );

        CREATE TABLE IF NOT EXISTS Bookings (
            booking_id        INTEGER  PRIMARY KEY AUTOINCREMENT,
            guest_id          INTEGER  NOT NULL,
            room_id           INTEGER  NOT NULL,
            check_in_date     TEXT     NOT NULL,
            check_out_date    TEXT     NOT NULL,
            check_in_year     INTEGER  NOT NULL,
            nights_stayed     INTEGER  NOT NULL CHECK(nights_stayed >= 1),
            num_guests        INTEGER  NOT NULL CHECK(num_guests BETWEEN 1 AND 4),
            booking_status    TEXT     NOT NULL DEFAULT 'Confirmed'
                              CHECK(booking_status IN
                              ('Confirmed','Checked-in','Checked-out','Cancelled','No-show')),
            payment_method    TEXT     NOT NULL
                              CHECK(payment_method IN
                              ('Credit Card','Debit Card','PayPal',
                               'Bank Transfer','Voucher','Crypto')),
            booking_channel   TEXT     NOT NULL
                              CHECK(booking_channel IN
                              ('Website','Phone','Walk-in','OTA','Travel Agent')),
            total_bill        REAL     NOT NULL CHECK(total_bill >= 0),
            FOREIGN KEY (guest_id) REFERENCES Guests(guest_id),
            FOREIGN KEY (room_id)  REFERENCES Rooms(room_id)
        );

        CREATE TABLE IF NOT EXISTS Stay_Services (
            booking_id      INTEGER  NOT NULL,
            service_id      INTEGER  NOT NULL,
            service_name    TEXT     NOT NULL,
            amount_charged  REAL     NOT NULL CHECK(amount_charged > 0),
            times_used      INTEGER  NOT NULL DEFAULT 1 CHECK(times_used >= 1),
            PRIMARY KEY (booking_id, service_id),
            FOREIGN KEY (booking_id) REFERENCES Bookings(booking_id)
        );

    """)
    print("Schema ready   5 tables created")
#below is the function which will insert the table one per table

def add_hotels(db_conn):
    """Inserts 15 hotels from HOTEL_DATA."""
    rows = [(name, city, stars, lo, hi) for name, city, stars, lo, hi in HOTEL_DATA]
    with db_conn:
        db_conn.executemany(
            "INSERT INTO Hotels (hotel_name, city, star_rating, rate_min, rate_max) VALUES (?,?,?,?,?)",
            rows
        )
    hotel_rows = db_conn.execute(
        "SELECT hotel_id, hotel_name, rate_min, rate_max FROM Hotels"
    ).fetchall()
    print(f"Hotels         {len(hotel_rows)} rows")
    return hotel_rows   # here it will return the hotel_id, name, rate_min, rate_max


def add_rooms(db_conn, hotel_rows):
    """
    Adds roughly 13-14 rooms per hotel (200 total) and the nightly rate is set within each hotel's own price band
    and then adjusted upward for nicer room types. (like single, double etc. )
    """
    type_multipliers = {
        "Single": 0.75, "Twin": 0.90, "Double": 1.0,
        "Deluxe Double": 1.3, "Suite": 1.8, "Penthouse": 2.5
    }
    type_occupancy = {
        "Single": 1, "Twin": 2, "Double": 2,
        "Deluxe Double": 2, "Suite": 3, "Penthouse": 4
    }

    all_room_rows = []
    for hotel_id, _, rate_min, rate_max in hotel_rows:
        for room_seq in range(13):
            rtype      = spin(ROOM_TYPES, ROOM_TYPE_WGTS)
            floor_num  = random.choice(ROOM_FLOORS)
            base_rate  = random.uniform(rate_min, rate_max)
            rate       = round(base_rate * type_multipliers[rtype], 2)
            occupancy  = type_occupancy[rtype]
            sea_view   = 1 if random.random() < 0.20 else 0
            room_num   = f"{floor_num}{str(room_seq + 1).zfill(2)}"
            all_room_rows.append(
                (hotel_id, room_num, floor_num, rtype, rate, occupancy, sea_view)
            )

    with db_conn:
        db_conn.executemany(
            """INSERT INTO Rooms
               (hotel_id, room_number, floor_number, room_type,
                nightly_rate, max_occupancy, has_sea_view)
               VALUES (?,?,?,?,?,?,?)""",
            all_room_rows
        )
    room_data = db_conn.execute(
        "SELECT room_id, nightly_rate FROM Rooms"
    ).fetchall()
    print(f"Rooms          {len(room_data)} rows")
    return room_data   # [(room_id, nightly_rate), ...]


def build_one_guest(used_emails, used_passports, force_no_email=False):
    """
    Certain guest records deliberately have missing values to give the appearance of real hotel data.
    About 8 percent of the guests do not have any email address in their record, usually due to phone booking or refusing to give
    any contact information, and about 15 percent have no passport number on record as domestic travelers in the UK are not always asked to enter a number.

Also, there is an absence of a satisfaction rating in 10 percent of the records since not all
guests are filling the feedback survey upon leaving. Whatever information is missing in all those is stored as
NULL so as to reflect the complete data of the real world correctly.

    """
    gender     = spin(GENDER_OPTIONS, GENDER_WEIGHTS)
    first      = pick_forename(gender)
    last       = random.choice(SURNAMES)
    birth_yr   = random.randint(1930, 2005)
    age_now    = 2024 - birth_yr
    nation     = random.choice(NATIONALITY_POOL)
    postcode   = random.choice(UK_POSTCODES) if nation == "United Kingdom" else None
    loyalty    = spin(LOYALTY_TIERS, LOYALTY_WEIGHTS)
    score      = random.randint(1, 10) if random.random() > 0.10 else None
    joined     = days_ago(10 * 365)
    phone      = f"07{random.randint(100,999)}{random.randint(100000,999999)}"

    if force_no_email or random.random() < 0.08:
        email = None
    else:
        email = make_email(first, last)
        while email in used_emails:
            email = make_email(first, last)
        used_emails.add(email)

    if random.random() < 0.15:
        passport = None
    else:
        passport = fake_passport()
        while passport in used_passports:
            passport = fake_passport()
        used_passports.add(passport)

    return (first, last, gender, birth_yr, age_now,
            nation, email, phone, postcode, passport,
            loyalty, score, joined)


def build_duplicate_guest(original_row, used_passports):
    """
    This will generate a duplicate guest record having a new passport
    number and none email address. It is an attempt to replicate a case scenario in which the same guest
    transacts using various mediums (e.g. direct and online travel agency), and in some cases, two different profiles are created in the hotel system.

    """
    new_passport = fake_passport()
    while new_passport in used_passports:
        new_passport = fake_passport()
    used_passports.add(new_passport)

    return (
        original_row[0], original_row[1],   # same first and last name from the array according to index number and so on for th below as well.
        original_row[2], original_row[3],   # same gender and birth_year
        original_row[4], original_row[5],   # same age and nationality
        None,                               # email NULL on duplicate profile
        original_row[7], original_row[8],   # same phone and postcode
        new_passport,                       # fresh passport number
        original_row[10], original_row[11], # same loyalty tier and score
        days_ago(3 * 365)                   # re-joined more recently
    )


def add_guests(db_conn):
    used_emails    = set()
    used_passports = set()

    genuine = [build_one_guest(used_emails, used_passports) for _ in range(970)]

    duplicates = [
        build_duplicate_guest(row, used_passports)
        for row in random.sample(genuine, 30)
    ]

    everyone = genuine + duplicates
    random.shuffle(everyone)

    with db_conn:
        db_conn.executemany(
            """INSERT INTO Guests
               (first_name, last_name, gender, birth_year, age,
                nationality, email_address, phone_number, postcode,
                passport_number, loyalty_tier, satisfaction_score,
                joined_programme)
               VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)""",
            everyone
        )
    guest_ids = [
        r[0] for r in db_conn.execute("SELECT guest_id FROM Guests").fetchall()
    ]
    print(f"Guests         {len(everyone)} rows  (970 genuine + 30 duplicate profiles)")
    return guest_ids


# This function handles all 1500 booking records for the database.
# I split it into two parts to keep things readable:
#   first build all the rows, then insert them in one go.
# check_out_date is always calculated from check_in_date plus the number
# of nights so the dates are never inconsistent with each other.
# total_bill at this stage only covers the room cost.
# The add_stay_services function adds extra charges on top later.

def make_single_booking(guest_ids, room_id_list, room_rate_map):
    chosen_guest  = random.choice(guest_ids)
    chosen_room   = random.choice(room_id_list)
    nightly_rate  = room_rate_map[chosen_room]

    # Most guests stay 1 to 3 nights, longer stays are less common
    number_of_nights = random.choices(
        [1, 2, 3, 4, 5, 6, 7, 10, 14],
        weights=[20, 25, 20, 15, 8, 4, 4, 3, 1]
    )[0]

    arrival_date   = days_ago(4 * 365)
    arrival_dt     = date.fromisoformat(arrival_date)
    departure_date = (arrival_dt + timedelta(days=number_of_nights)).isoformat()

    # Most bookings are for 1 or 2 guests, groups of 3 or 4 are rare
    people_in_room = random.choices([1, 2, 3, 4], weights=[40, 40, 12, 8])[0]

    room_only_bill = round(number_of_nights * nightly_rate, 2)

    return (
        chosen_guest,
        chosen_room,
        arrival_date,
        departure_date,
        # check_in_year stored separately as INTERVAL data
        int(arrival_date[:4]),
        number_of_nights,
        people_in_room,
        spin(BOOKING_STATUSES, BOOKING_STATUS_WGT),
        spin(PAYMENT_OPTIONS,  PAYMENT_WEIGHTS),
        spin(BOOKING_CHANNELS, CHANNEL_WEIGHTS),
        room_only_bill,
    )


def add_bookings(db_conn, guest_ids, room_data):

    room_id_list  = [r[0] for r in room_data]
    room_rate_map = {r[0]: r[1] for r in room_data}

    # Build every row first, then insert all at once
    # One big insert is faster than inserting one row at a time
    all_booking_rows = [
        make_single_booking(guest_ids, room_id_list, room_rate_map)
        for _ in range(1500)
    ]

    with db_conn:
        db_conn.executemany(
            """INSERT INTO Bookings
               (guest_id, room_id, check_in_date, check_out_date,
                check_in_year, nights_stayed, num_guests,
                booking_status, payment_method, booking_channel, total_bill)
               VALUES (?,?,?,?,?,?,?,?,?,?,?)""",
            all_booking_rows
        )

    # Fetch back the IDs so the next function knows which bookings exist
    all_booking_ids = [
        r[0] for r in
        db_conn.execute("SELECT booking_id FROM Bookings").fetchall()
    ]

    print(f"Bookings       {len(all_booking_ids)} rows")
    return all_booking_ids


def add_stay_services(db_conn):
    """
    Charges in-stay services to bookings that are checked out.
    Primary key is a compound key (booking_id, service_id) such that.
    same service type cannot be repeated in a single booking.
    Calculates additional charges and re-calculates total bill in a single batch.
    """
    checked_out = db_conn.execute(
        "SELECT booking_id FROM Bookings WHERE booking_status = 'Checked-out'"
    ).fetchall()
    checked_out_ids = [r[0] for r in checked_out]

    service_rows      = []
    extra_per_booking = {}   # booking_id -> extra service charges

    for b_id in checked_out_ids:
        how_many  = random.choices([1,2,3,4], weights=[35,35,20,10])[0]
        chosen    = random.sample(SERVICE_CATALOGUE, how_many)

        for svc_id, svc_name, price_lo, price_hi in chosen:
            charge    = round(random.uniform(price_lo, price_hi), 2)
            times     = random.choices([1,2,3], weights=[65,25,10])[0]
            total_svc = round(charge * times, 2)
            service_rows.append((b_id, svc_id, svc_name, charge, times))
            extra_per_booking[b_id] = extra_per_booking.get(b_id, 0.0) + total_svc

    with db_conn:
        db_conn.executemany(
            """INSERT OR IGNORE INTO Stay_Services
               (booking_id, service_id, service_name, amount_charged, times_used)
               VALUES (?,?,?,?,?)""",
            service_rows
        )

    # Add the service charges on top of the room-only total_bill
    with db_conn:
        db_conn.executemany(
            "UPDATE Bookings SET total_bill = ROUND(total_bill + ?, 2) WHERE booking_id = ?",
            [(extra, bid) for bid, extra in extra_per_booking.items()]
        )

    svc_count = db_conn.execute("SELECT COUNT(*) FROM Stay_Services").fetchone()[0]
    print(f"Stay_Services  {svc_count} rows  (composite PK on booking_id + service_id)")


# Prints row counts for each table and shows how many NULLs were added on purpose and check all the data we have inserted and created

def print_summary(db_conn):

    print("\nRow counts:")
    for table_name in ["Hotels", "Rooms", "Guests", "Bookings", "Stay_Services"]:
        count = db_conn.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        print(f"  {table_name} : {count} rows")

    print("\nDeliberate missing values:")
    checks = [
        ("Email missing",      "SELECT COUNT(*) FROM Guests WHERE email_address IS NULL"),
        ("Passport missing",   "SELECT COUNT(*) FROM Guests WHERE passport_number IS NULL"),
        ("No satisfaction score", "SELECT COUNT(*) FROM Guests WHERE satisfaction_score IS NULL"),
    ]
    for label, query in checks:
        count = db_conn.execute(query).fetchone()[0]
        print(f"  {label} : {count}")

    print(f"  Duplicate guests : 30 (added on purpose)")


def main():
    target = "hotel.db"

    if os.path.exists(target):
        os.remove(target)
        print("Old database cleared.")

    db_conn = sqlite3.connect(target)
    db_conn.execute("PRAGMA foreign_keys = ON")
    db_conn.execute("PRAGMA journal_mode = WAL")
    print(f"Connected to {target}\n")

    create_all_tables(db_conn)

    hotel_rows  = add_hotels(db_conn)
    room_data   = add_rooms(db_conn, hotel_rows)
    guest_ids   = add_guests(db_conn)
    booking_ids = add_bookings(db_conn, guest_ids, room_data)
    add_stay_services(db_conn)

    print_summary(db_conn)

    db_conn.close()
    print(f"\nAll data   saved to {target}")


if __name__ == "__main__":
    main()

Old database cleared.
Connected to hotel.db

Schema ready   5 tables created
Hotels         15 rows
Rooms          195 rows
Guests         1000 rows  (970 genuine + 30 duplicate profiles)
Bookings       1500 rows
Stay_Services  1832 rows  (composite PK on booking_id + service_id)

Row counts:
  Hotels : 15 rows
  Rooms : 195 rows
  Guests : 1000 rows
  Bookings : 1500 rows
  Stay_Services : 1832 rows

Deliberate missing values:
  Email missing : 116
  Passport missing : 139
  No satisfaction score : 94
  Duplicate guests : 30 (added on purpose)

All data   saved to hotel.db
